In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import warnings
warnings.filterwarnings('ignore')

GLOBAL_TRUTH = 0.8

algorithms = {
    'epsilon_greedy': {'type': 'context_free', 'params': {'strategy_type': 'epsilon_greedy', 'epsilon': 0.1}},
    'ucb': {'type': 'context_free', 'params': {'strategy_type': 'ucb', 'c_param': 1.0}},
    'thompson': {'type': 'context_free', 'params': {'strategy_type': 'thompson'}},
    'lin_epsilon_greedy': {'type': 'contextual', 'params': {'strategy_type': 'lin_epsilon_greedy', 'epsilon': 0.1}},
    'linucb': {'type': 'contextual', 'params': {'strategy_type': 'linucb', 'alpha': 1.5}},
    'lin_thompson': {'type': 'contextual', 'params': {'strategy_type': 'lin_thompson', 'sigma': 1.0}}
}


class Agent:
    def __init__(self, node_id, initial_opinion):
        self.id = node_id
        self.opinion = initial_opinion
        self.neighbors = []
        self.is_leader = False

    def set_neighbors(self, neighbors):
        self.neighbors = neighbors

    def get_neighbors_opinions(self):
        if not self.neighbors:
            return [self.opinion]
        return [n.opinion for n in self.neighbors]

    def apply_strategy(self, strategy_id):
        """
        Применяет одну из 5 стратегий:
        0: Stubborn - оставить мнение без изменений
        1: Average - полное усреднение с соседями
        2: Compromise - сдвиг на 30% к среднему соседей
        3: Drift Up - прибавить 0.05, если текущее <= 0.95
        4: Drift Down - отнять 0.05, если текущее >= 0.05
        """
        current = self.opinion
        neighbors_ops = self.get_neighbors_opinions()
        avg_neighbors = np.mean(neighbors_ops)

        if not self.is_leader:
            if strategy_id == 0:  # Stubborn
                new_opinion = current
            elif strategy_id == 1:  # Average
                new_opinion = avg_neighbors
            elif strategy_id == 2:  # Compromise
                alpha = 0.3
                new_opinion = (1 - alpha) * current + alpha * avg_neighbors
            elif strategy_id == 3:  # Drift Up
                if current <= 0.95:
                    new_opinion = current + 0.05
                else:
                    new_opinion = current
            elif strategy_id == 4:  # Drift Down
                if current >= 0.05:
                    new_opinion = current - 0.05
                else:
                    new_opinion = current
            else:
                new_opinion = current
        else:
            new_opinion = current

        self.opinion = np.clip(new_opinion, 0, 1)
        return self.opinion



class ContextFreeBandit:
    def __init__(self, n_actions=5, strategy_type='ucb', epsilon=0.1, c_param=1.0):
        self.n_actions = n_actions
        self.strategy_type = strategy_type
        self.epsilon = epsilon
        self.c_param = c_param

        self.counts = np.zeros(n_actions)
        self.values = np.zeros(n_actions)

    def select_action(self):
        if self.strategy_type == 'epsilon_greedy':
            if np.random.rand() < self.epsilon:
                return np.random.randint(self.n_actions)
            else:
                if np.all(self.counts == 0):
                    return np.random.randint(self.n_actions)
                return np.argmax(self.values)

        elif self.strategy_type == 'ucb':
            if np.sum(self.counts) < self.n_actions:
                for i in range(self.n_actions):
                    if self.counts[i] == 0:
                        return i

            ucb_values = []
            total_counts = np.sum(self.counts)
            for i in range(self.n_actions):
                confidence = self.c_param * np.sqrt(np.log(total_counts) / self.counts[i])
                ucb_values.append(self.values[i] + confidence)
            return np.argmax(ucb_values)

        elif self.strategy_type == 'thompson':
            samples = []
            for i in range(self.n_actions):
                if self.counts[i] == 0:
                    samples.append(np.inf)
                else:
                    variance = 1.0 / (self.counts[i] + 1)
                    samples.append(np.random.normal(self.values[i], np.sqrt(variance)))
            return np.argmax(samples)

        return np.random.randint(self.n_actions)

    def update(self, action, reward):
        self.counts[action] += 1
        self.values[action] += (reward - self.values[action]) / self.counts[action]




class ContextualBandit:
    def __init__(self, n_actions=5, d_context=4, strategy_type='linucb',
                 epsilon=0.1, alpha=1.0, sigma=1.0):
        self.n_actions = n_actions
        self.d = d_context
        self.strategy_type = strategy_type
        self.epsilon = epsilon
        self.alpha = alpha
        self.sigma = sigma

        self.A = [np.eye(d_context) for _ in range(n_actions)]
        self.b = [np.zeros(d_context) for _ in range(n_actions)]
        self.A_inv = [np.eye(d_context) for _ in range(n_actions)]

    def get_theta(self, action):
        return self.A_inv[action].dot(self.b[action])

    def select_action(self, context):
        context = np.array(context)

        if self.strategy_type == 'lin_epsilon_greedy':
            if np.random.rand() < self.epsilon:
                return np.random.randint(self.n_actions)
            else:
                p_values = [context.dot(self.get_theta(i)) for i in range(self.n_actions)]
                return np.argmax(p_values)

        elif self.strategy_type == 'linucb':
            p_values = []
            for i in range(self.n_actions):
                theta = self.get_theta(i)
                p = context.dot(theta)
                variance_term = np.sqrt(context.dot(self.A_inv[i]).dot(context))
                ucb = p + self.alpha * variance_term
                p_values.append(ucb)
            return np.argmax(p_values)

        elif self.strategy_type == 'lin_thompson':
            samples = []
            for i in range(self.n_actions):
                theta = self.get_theta(i)
                cov = (self.sigma ** 2) * self.A_inv[i]
                try:
                    theta_sample = np.random.multivariate_normal(theta, cov)
                    samples.append(context.dot(theta_sample))
                except:
                    samples.append(context.dot(theta))
            return np.argmax(samples)

        return np.random.randint(self.n_actions)

    def update(self, action, context, reward):
        x = context.reshape(-1, 1)
        self.A[action] += x.dot(x.T)
        self.b[action] += reward * x.flatten()
        self.A_inv[action] = np.linalg.inv(self.A[action])




def get_context(agent, global_truth):
    dist_to_truth = abs(agent.opinion - global_truth)
    neighbor_ops = agent.get_neighbors_opinions()
    neighbor_variance = np.var(neighbor_ops) if len(neighbor_ops) > 1 else 0.0

    return np.array([1.0, agent.opinion, dist_to_truth, neighbor_variance])




def run_multiple_simulations(n_runs=10, n_steps=100, agents=None, global_truth=GLOBAL_TRUTH):
    results_pool = {}
    n_nodes = len(agents)


    STRATEGIES_MAP = ['Стуб', 'Аве', 'Комп', '+0.05', '-0.05']

    for algo_name, config in algorithms.items():
        print(f"\nАлгоритм: {algo_name} | Запуск {n_runs} симуляций...")

        all_opinions = []
        all_rewards = []
        strategy_counts = []
        final_opinions_array = []

        for r in range(1, n_runs + 1):
            base_seed = 42 + r

            for i, agent in enumerate(agents):
                agent.opinion = np.random.uniform(low=0.1, high=0.9)

            bandits = []
            for i, agent in enumerate(agents):
                if config['type'] == 'context_free':
                    params = config['params'].copy()
                    params['n_actions'] = 5
                    bandits.append(ContextFreeBandit(**params))
                else:
                    params = config['params'].copy()
                    params['n_actions'] = 5
                    bandits.append(ContextualBandit(d_context=4, **params))

            opinions_run = []
            rewards_run = []
            step_strategy_counts = np.zeros(5)
            final_opinions_this_run = []

            for t in range(n_steps):
                step_rewards = []
                new_opinions = []

                for i, agent in enumerate(agents):
                    bandit = bandits[i]

                    if config['type'] == 'context_free':
                        action = bandit.select_action()
                        context = None
                    else:
                        context = get_context(agent, global_truth)

                        if config['type'] == 'contextual' and np.random.rand() < 0.05:
                             context += np.random.normal(0, 0.01, size=context.shape)

                        action = bandit.select_action(context)

                    new_op = agent.apply_strategy(action)
                    new_opinions.append(new_op)

                    reward = -abs(new_op - global_truth)
                    step_rewards.append(reward)

                    if config['type'] == 'context_free':
                        bandit.update(action, reward)
                    else:
                        context_for_update = get_context(agent, global_truth)
                        bandit.update(action, context_for_update, reward)

                    step_strategy_counts[action] += 1

                for i, agent in enumerate(agents):
                    agent.opinion = new_opinions[i]

                avg_reward = np.mean(step_rewards)
                avg_followers = np.mean([a.opinion for a in agents])

                opinions_run.append(avg_followers)
                rewards_run.append(avg_reward)

                if t == n_steps - 1:
                    final_opinions_this_run = [a.opinion for a in agents]
                    strategy_counts.append(step_strategy_counts.copy())

            all_opinions.append(opinions_run)
            all_rewards.append(rewards_run)
            final_opinions_array.append(final_opinions_this_run)

        strategy_counts = np.array(strategy_counts)
        final_opinions_array = np.array(final_opinions_array)
        all_opinions = np.array(all_opinions)
        all_rewards = np.array(all_rewards)

        results_pool[algo_name] = {
            'mean_opinions': np.mean(all_opinions, axis=0),
            'mean_rewards': np.mean(all_rewards, axis=0),
            'std_opinions': np.std(all_opinions, axis=0),
            'std_rewards': np.std(all_rewards, axis=0),
            'final_opinion': float(np.mean(all_opinions, axis=0)[-1]),
            'total_reward': float(np.sum(np.mean(all_rewards, axis=0))),
            'type': config['type'],
            'final_opinions_node_level': np.mean(final_opinions_array, axis=0).tolist(),
            'avg_strategy_counts': np.mean(strategy_counts, axis=0).tolist()
        }


        strat_str = ', '.join([f'{STRATEGIES_MAP[i]}:{results_pool[algo_name]["avg_strategy_counts"][i]:.2f}' for i in range(5)])
        print(f"  Финальное среднее мнение: {results_pool[algo_name]['final_opinion']:.4f}")
        print(f"     Средняя частота стратегий: {strat_str}")

    return results_pool



def generate_synthetic_graph(n_agents, model='ba', param=4):

    if model == 'ba':

        G = nx.barabasi_albert_graph(n_agents, m=param, seed=42)

    elif model == 'er':
        p = param / n_agents
        G = nx.erdos_renyi_graph(n_agents, p=p, seed=42)

    elif model == 'ws':
        G = nx.watts_strogatz_graph(n_agents, k=param, p=0.1, seed=42)

    else:

        G = nx.complete_graph(n_agents)

    return G

def run_full_simulation(n_runs=10, n_steps=100, n_nodes=50, seed=42):
    np.random.seed(seed)
    G = generate_synthetic_graph(n_nodes, model='er', param=4)



    agents = []
    for i in range(n_nodes):
        initial_mind = np.random.rand()
        agents.append(Agent(i, initial_mind))

    for i in range(n_nodes):
        neighbors_ids = list(G.neighbors(i))
        neighbors_agents = [agents[j] for j in neighbors_ids]
        agents[i].set_neighbors(neighbors_agents)

    global_truth = GLOBAL_TRUTH

    print(f"Количество агентов: {n_nodes}")
    print(f"Количество шагов: {n_steps}")
    print(f"Количество run: {n_runs}")
    print(f"Базовое seed: {seed}")
    print(f"Внешняя истина: {global_truth}")
    print("=" * 60)

    results = run_multiple_simulations(
        n_runs=n_runs,
        n_steps=n_steps,
        agents=agents,
        global_truth=global_truth
    )

    return results, global_truth, agents, G




def plot_results(results, global_truth, n_runs=10):


    colors_cf = {'epsilon_greedy': 'blue', 'ucb': 'orange', 'thompson': 'green'}
    colors_ctx = {'lin_epsilon_greedy': 'cyan', 'linucb':  'khaki', 'lin_thompson': 'lightgreen'}

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))


    for algo, data in results.items():
        color = colors_cf.get(algo, colors_ctx.get(algo, 'gray'))
        linestyle = '-' if data['type'] == 'context_free' else '--'
        label = algo.replace('_', ' ').title()

        cumulative = np.cumsum(data['mean_rewards'])
        ax1.plot(cumulative, label=label, color=color, linestyle=linestyle,
                linewidth=2.5, alpha=0.9)

    ax1.set_xlabel('Шаг')
    ax1.set_ylabel('Накопленная награда')
    ax1.legend(loc='lower right', fontsize=10)
    ax1.grid(True, alpha=0.3)


    for algo, data in results.items():
        color = colors_cf.get(algo, colors_ctx.get(algo, 'gray'))
        linestyle = '-' if data['type'] == 'context_free' else '--'
        label = algo.replace('_', ' ').title()

        ax2.plot(data['mean_opinions'], label=label, color=color,
                linestyle=linestyle, linewidth=2.5, alpha=0.9)

        se_error = data['std_opinions'] / np.sqrt(n_runs)
        ax2.fill_between(range(len(se_error)),
                        data['mean_opinions'] - se_error,
                        data['mean_opinions'] + se_error,
                        color=color, alpha=0.2)

    ax2.axhline(global_truth, color='red', linestyle=':', linewidth=3,
               label='Истина (θ)')
    ax2.set_xlabel('Шаг')
    ax2.set_ylabel('Среднее мнение')
    ax2.legend(loc='best', fontsize=10)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('combined_results_cumulative_vs_opinion.png', dpi=300, bbox_inches='tight')
    plt.show()


def plot_graph_heatmaps(results, agents, G, global_truth):
    cmap = plt.cm.RdYlBu_r
    n_nodes = len(agents)

    max_neighbors = 1
    if n_nodes > 0:
        neighbor_counts = [len(list(G.neighbors(i))) for i in range(n_nodes)]
        if neighbor_counts:
            max_neighbors = max(neighbor_counts) + 1

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.ravel()

    for i, (algo_name, data) in enumerate(results.items()):
        if i >= 6:
            break

        if data['type'] == 'context_free':
            color = {'epsilon_greedy': 'red', 'ucb': 'blue', 'thompson': 'green'}.get(algo_name, 'gray')
        else:
            color = {'lin_epsilon_greedy': 'orange', 'linucb': 'purple', 'lin_thompson': 'cyan'}.get(algo_name, 'gray')

        if 'final_opinions_node_level' in data and isinstance(data['final_opinions_node_level'], list) and len(data['final_opinions_node_level']) == n_nodes:
            node_values = data['final_opinions_node_level']
        else:
            node_values = [data['final_opinion']] * n_nodes

        ax = axes[i]
        pos = nx.spring_layout(G, k=2/np.sqrt(n_nodes), iterations=100, seed=42)

        sizes = [300 + 50 * (len(list(G.neighbors(j))) // max_neighbors) for j in range(n_nodes)]

        norm = Normalize(vmin=0, vmax=1)

        nodes = nx.draw_networkx_nodes(G, pos, node_size=sizes,
                                       node_color=[norm(val) for val in node_values],
                                       cmap=cmap, alpha=0.8, ax=ax,
                                       edgecolors='white', linewidths=0.5,
                                       nodelist=range(n_nodes))

        nx.draw_networkx_edges(G, pos, edgelist=G.edges(), width=1, alpha=0.3,
                              edge_color='#cccccc', ax=ax)

        if nodes is not None:
            cbar = plt.colorbar(nodes, ax=ax, shrink=0.7, pad=0.05, label='Мнение агента')

        labels = {j: str(j) for j in range(n_nodes)}
        nx.draw_networkx_labels(G, pos, labels=labels, font_size=6, font_weight='bold', ax=ax)

        mean_val = np.mean(node_values)

        title_parts = [f'{algo_name.replace("_", " ").title()} (Mean: {mean_val:.3f})']
        ax.set_title(' '.join(title_parts), fontsize=10, fontweight='bold', pad=10)
        ax.axis('off')

        filename = f'heatmap_{algo_name}_run1.png'
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        print(f" Сохранён файл: {filename}")

    plt.tight_layout()
    plt.show()



def print_summary(results, global_truth, n_runs=10):
    strategies_map = ['Стуб', 'Аве', 'Комп', '+0.05', '-0.05']

    print(f"{'Алгоритм':<20} {'Тип':<10} {'Мнение':<10} {'SE':<8} {'Награда':<12} {'Ошибка':<10} {'Ср. Стратегии':<40}")
    print("-" * 120)

    sorted_results = sorted(results.items(), key=lambda x: abs(x[1]['final_opinion'] - global_truth))

    for algo, data in sorted_results:
        final_op = data['final_opinion']
        err_abs = abs(final_op - global_truth)
        total_rev = data['total_reward']
        algo_type = "Без контекста" if data['type'] == 'context_free' else "С контекстом"
        se = data['std_opinions'][-1] / np.sqrt(n_runs)

        strat_str = ', '.join([f'{strategies_map[i]}:{data["avg_strategy_counts"][i]:.2f}' for i in range(5)])

        print(f"{algo:<20} {algo_type:<10} {final_op:<10.4f} {se:<8.4f} {total_rev:<12.2f} {err_abs:<10.4f} {strat_str}")

    print("=" * 120)

    best_algo = sorted_results[0][0]
    best_error = abs(sorted_results[0][1]['final_opinion'] - global_truth)
    best_se = sorted_results[0][1]['std_opinions'][len(sorted_results[0][1]['std_opinions']) - 1] / np.sqrt(n_runs)
    print(f"\n Лучший алгоритм: {best_algo} (ошибка: {best_error:.4f} ± {best_se:.4f})")
    print("=" * 120)


if __name__ == "__main__":
    N_RUNS = 250
    N_STEPS = 300
    N_NODES = 50
    BASE_SEED = 42

    results, global_truth, agents, G = run_full_simulation(
        n_runs=N_RUNS,
        n_steps=N_STEPS,
        n_nodes=N_NODES,
        seed=BASE_SEED
    )

    plot_results(results, global_truth, n_runs=N_RUNS)
    plot_graph_heatmaps(results, agents, G, global_truth)
    print_summary(results, global_truth, n_runs=N_RUNS)